In [ ]:
import math, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

**RULES**

R1. The split is the same as Day 12: test_size=0.2, random_state=42, stratify=y.

R2. The scaler is fitted on the TRAINING rows only, then applied to the test rows.

R3. Training loader: shuffle=True. Test loader: shuffle=False.

R4. Every tensor handed to a model is float32; features have no requires_grad.

R5. After building, check shapes, row counts and churn counts. If a check fails, stop.



In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["churn"] = (df["Churn"] == "Yes").astype(int)

num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = ["Contract", "InternetService", "PaymentMethod",
        "PaperlessBilling", "Partner", "Dependents", "TechSupport"]
X = pd.get_dummies(df[num + cat], columns=cat, dtype=int)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X.shape, X_train.shape, X_test.shape)
print("train churners", y_train.sum(), round(y_train.mean(), 4),
        "| test churners", y_test.sum(), round(y_test.mean(), 4))


(7043, 23) (5634, 23) (1409, 23)
train churners 1495 0.2654 | test churners 374 0.2654


In [4]:
X_train_m = X_train.median()

X_train["TotalCharges"] = X_train["TotalCharges"].fillna(X_train_m["TotalCharges"])
X_test["TotalCharges"] = X_test["TotalCharges"].fillna(X_train_m["TotalCharges"])

In [6]:
print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

0
0


- We fill the null values of train and test set with median oftraining rows because a median over the whole table would use the test rows’ values to fill the training rows. So information from the test set would flow into training, and your test score would be slightly too optimistic.

- Use the train median because the whole-table median would include test rows, leaking test information into training.

In [9]:
scaler = StandardScaler().fit(X_train)
a = scaler.transform(X_train)
b = scaler.transform(X_test)
Xt = torch.tensor(a, dtype=torch.float32)
yt = torch.tensor(y_train.values, dtype=torch.float32)
Xe = torch.tensor(b, dtype=torch.float32)
ye = torch.tensor(y_test.values, dtype=torch.float32)
print(Xt.shape, Xt.dtype, yt.shape, yt.dtype, Xe.shape, ye.shape, Xt.requires_grad)
print("train means:", a.mean(0).round(3)[:4], "stds:", a.std(0).round(3)[:4])
print("test means: ", b.mean(0).round(3)[:4])
print("churn rate train", round(yt.mean().item(), 4), "test", round(ye.mean().item(), 4))


torch.Size([5634, 23]) torch.float32 torch.Size([5634]) torch.float32 torch.Size([1409, 23]) torch.Size([1409]) False
train means: [-0. -0. -0.  0.] stds: [1. 1. 1. 1.]
test means:  [-0.023 -0.028 -0.043 -0.016]
churn rate train 0.2654 test 0.2654


- The test means are NOT zero and it is not a bug because they were scaled using training set's centre points, so the test mean's will be off instead of zero is completely alright which it tells you that the model is scaled using only training set.

In [11]:
class PairDataset(Dataset):
  def __init__(self, x, y):
    self.x = x
    self.y = y

  def __len__(self):
    return len(self.x)

  def __getitem__(self, i):
    return self.x[i], self.y[i]
train_ds = PairDataset(Xt, yt)
test_ds = PairDataset(Xe, ye)

print(len(train_ds), len(test_ds))
x0, y0 = train_ds[0]
print(x0.shape, x0.dtype, y0)

5634 1409
torch.Size([23]) torch.float32 tensor(0.)


- A single sample from the dataset represents one individual customer.
- The shape of x0 is 23 which is a 1-dimensional vector containing exactly 23 numeric values. Each of these 23 values is a scaled, standardized feature that describes the customer's account attributes.
- y0 is a single number because it the target variable containing customer's churn data. So the first customer's churn is 0 (No)

In [13]:
torch.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
print(len(train_loader), len(test_loader))

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape, xb.dtype, yb.dtype)
sizes = [len(b[1]) for b in train_loader]
print(sizes[:3], sizes[-3:], sum(sizes))

89 12
torch.Size([64, 23]) torch.Size([64]) torch.float32 torch.float32
[64, 64, 64] [64, 64, 2] 5634


- There are 89 bathces and the last one has rows which is the last remaining rows left. (5634 = 88 * 64 + 2). 88 full batches plus 1 small batch = 89

In [14]:
for epoch in range(3):
  seen, churners, first = 0, 0, None
  for i, (xb, yb) in enumerate(train_loader):
    seen += len(yb)
    churners += int(yb.sum().item())
    if i == 0:
      first = yb[:8].int().tolist()
  print("epoch", epoch, seen, churners, first)

epoch 0 5634 1495 [0, 0, 0, 0, 0, 0, 0, 0]
epoch 1 5634 1495 [0, 0, 1, 0, 0, 1, 0, 0]
epoch 2 5634 1495 [0, 0, 0, 0, 0, 0, 0, 0]


- This proves that in every loop through the epoch the seen and churners are the same, tells you that the data pipeline isn't losing any rows in transit, duplicating any records, or dropping the remainder batch.
- Changing the first list proves that the pipeline actively shuffling.

In [16]:
train_loader_fixed = DataLoader(train_ds, batch_size=64, shuffle=False)
xb, yb = next(iter(train_loader_fixed))
w = torch.zeros(Xt.shape[1], requires_grad=True)
loss = ((xb @ w - yb) ** 2).mean()
loss.backward()
print(w.grad.shape, round(loss.item(), 4), round(yb.mean().item(), 4))

torch.Size([23]) 0.25 0.25


- The loss exactly equal to the churn rate of this batch because the weights are all set to zero(0), then all the predictions is 0 for every row.
- Each single y² is 0 or 1, but the mean over 64 rows is a fraction.
- with w = 0, every prediction is 0, so loss = mean((0 − y)²) = mean(y²). Because y is only 0 or 1, y² = y, so loss = mean(y) = the fraction of churners in the batch. Here that is 16 churners out of 64 = 0.25.

In [18]:
import torch
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

train_loader_fixed = DataLoader(train_ds, batch_size=64, shuffle=False)
xb, yb = next(iter(train_loader_fixed))

xb = xb.to(device)
yb = yb.to(device)
print(xb.device)

w = torch.zeros(Xt.shape[1], requires_grad=True, device=device)
loss = ((xb @ w - yb) ** 2).mean()
loss.backward()
print(w.grad.shape, round(loss.item(), 4), round(yb.mean().item(), 4))

cuda:0
torch.Size([23]) 0.25 0.25


- The three properties to check on every tensor is dtype (mean(): could not infer output dtype (an int tensor)), device (Expected all tensors to be on the same device), and shape(mat1 and mat2 shapes cannot be multiplied).

- require_grads=True do  automatic differentiation engine (torch.autograd) that it should track and record all mathematical operations performed on that tensor.

- If you call backward() without zeroing the gradient the old derivatives continuously accumulate and pile up inside the .grad attribute, causing your optimization steps to explode or overshoot the minimum.

- Difference between dataset and dataloader is dataset returns the rows while dataloader serves the batch by initializing batch_size, shuffle, drop_last.

- Epoch is one complete pass where the model evaluates every single row in the dataset, bacth is one small slice of rows handed to the model at one time, batch_size is the size of the exact batch that handed to the model at a time. There are 88 batches plus 2 remaining rows in 5634 because 88 full batches plus 1 small batch of 2 rows = 89 batches.

- shuffle is True for training rows because training reorganizes the row sequences at the start of every epoch, forcing the neural network to learn genuine underlying patterns instead of accidentally memorizing the exact order of the data sheet. shuffle is False for test set because we are simply scoring a finalized model.

** Conclusion **
- network weights and gradients are floating point, requires_grad only works on floats, and NumPy gives float64 by default while PyTorch layers expect float32. Mixing them raises errors.

- We always scaled on training set so it learns the underlying pattern and perform well and also there will be no data leakage.

- We always need to zero the gradient before we call the backward so the old derivatives won't pile up into grad and cause to overshoot.

- Always set shuffle = True for training set so the model can able to understand and learn the underlying pattern of the dataset.

- A Dataset is a storage box that holds your data and looks up individual customer samples one by one, while a DataLoader is the active engine that wraps around that box to shuffle the cards and serve them out in organized mini-batches.

- Autograd will perform automatic differentiation engine that records a history of all mathematical operations performed on your tensors to automatically calculate gradients  (derivatives) during a backpropagation pass